In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import os
import glob
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import random

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(SimpleUNet, self).__init__()
         
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)
        
        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        
        self.out = nn.Conv2d(16, 3, 1)
        
    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        
        b = self.bottleneck(p2)

        u2 = self.up2(b)

        if u2.shape != e2.shape:
            u2 = torch.nn.functional.interpolate(u2, size=e2.shape[2:], mode='bilinear')
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        if u1.shape != e1.shape:
            u1 = torch.nn.functional.interpolate(u1, size=e1.shape[2:], mode='bilinear')
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        return self.out(d1)




In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, num_classes=5, img_channels=3):
        super(Generator, self).__init__()
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        
        self.init_size = 16
        self.label_embedding = nn.Embedding(num_classes, latent_dim)
        
        self.l1 = nn.Sequential(nn.Linear(latent_dim * 2, 128 * self.init_size ** 2))
        
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.1),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, img_channels, 3, stride=1, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        label_embed = self.label_embedding(labels)
        gen_input = torch.cat((label_embed, z), -1)
        
        out = self.l1(gen_input)
        out = out.view(out.shape[0], 128, self.init_size, self.init_size)
        img = self.conv_blocks(out)
        return img

class Discriminator(nn.Module):
    def __init__(self, num_classes=5, img_channels=3):
        super(Discriminator, self).__init__()
        
        def discriminator_block(in_filters, out_filters, bn=True):
            block = [nn.Conv2d(in_filters, out_filters, 3, 2, 1),
                    nn.LeakyReLU(0.2, inplace=True),
                    nn.Dropout2d(0.3)]
            if bn:
                block.append(nn.BatchNorm2d(out_filters, 0.8))
            return block
        
        self.model = nn.Sequential(
            *discriminator_block(img_channels, 16, bn=False),
            *discriminator_block(16, 32),
            *discriminator_block(32, 64),
            *discriminator_block(64, 128),
        )
        
        ds_size = 64 // 2 ** 4
        self.adv_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, 1), nn.Sigmoid())
        self.aux_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, num_classes), nn.Softmax(dim=1))
    
    def forward(self, img):
        out = self.model(img)
        out = out.view(out.shape[0], -1)
        validity = self.adv_layer(out)
        label = self.aux_layer(out)
        return validity, label




In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    gan_model_path = r"C:\Users\Administrator\OneDrive\Desktop\diana\Gan Anomaly\final_hybrid_model.pth"
    unet_model_path = r"C:\Users\Administrator\OneDrive\Desktop\diana\Gan Anomaly\2_unet_trained.pth"
    
    image_path = r"C:\Users\Administrator\OneDrive\Desktop\diana\Gan Anomaly\tzh_wear\adhasive\pic5_3_135.png"
    
    if not os.path.exists(gan_model_path):
        print(f"no model: {gan_model_path}")
        return
    
    if not os.path.exists(image_path):
        print(f"no imgs: {image_path}")
        return
    

    class FixedSimpleUNet(nn.Module):
        def __init__(self, in_channels=3, out_channels=1):
            super(FixedSimpleUNet, self).__init__()

            self.enc1 = nn.Sequential(
                nn.Conv2d(in_channels, 16, 3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU(),
                nn.Conv2d(16, 16, 3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU()
            )
            self.pool1 = nn.MaxPool2d(2)
            
            self.enc2 = nn.Sequential(
                nn.Conv2d(16, 32, 3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(),
                nn.Conv2d(32, 32, 3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU()
            )
            self.pool2 = nn.MaxPool2d(2)

            self.bottleneck = nn.Sequential(
                nn.Conv2d(32, 64, 3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(),
                nn.Conv2d(64, 64, 3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU()
            )

            self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
            self.dec2 = nn.Sequential(
                nn.Conv2d(64, 32, 3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(),
                nn.Conv2d(32, 32, 3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU()
            )
            
            self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
            self.dec1 = nn.Sequential(
                nn.Conv2d(32, 16, 3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU(),
                nn.Conv2d(16, 16, 3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU()
            )
            
            self.out = nn.Conv2d(16, out_channels, 1)
            
        def forward(self, x):
            e1 = self.enc1(x)
            p1 = self.pool1(e1)
            
            e2 = self.enc2(p1)
            p2 = self.pool2(e2)

            b = self.bottleneck(p2)
            
            u2 = self.up2(b)
            if u2.shape != e2.shape:
                u2 = torch.nn.functional.interpolate(u2, size=e2.shape[2:], mode='bilinear')
            d2 = self.dec2(torch.cat([u2, e2], dim=1))
            
            u1 = self.up1(d2)
            if u1.shape != e1.shape:
                u1 = torch.nn.functional.interpolate(u1, size=e1.shape[2:], mode='bilinear')
            d1 = self.dec1(torch.cat([u1, e1], dim=1))
            
            return self.out(d1)
    
    unet = FixedSimpleUNet(in_channels=3, out_channels=1).to(device)
    generator = Generator(latent_dim=100, num_classes=5, img_channels=3).to(device)
    discriminator = Discriminator(num_classes=5, img_channels=3).to(device)

    checkpoint = torch.load(gan_model_path, map_location=device)
    
    generator_keys = ['generator', 'G', 'generator_state_dict']
    discriminator_keys = ['discriminator', 'D', 'discriminator_state_dict']

    generator_loaded = False
    for key in generator_keys:
        if key in checkpoint:
            try:
                generator.load_state_dict(checkpoint[key])
                print(f"Generator загружен из ключа: {key}")
                generator_loaded = True
                break
            except:
                continue
    
    if not generator_loaded:
        try:
            generator.load_state_dict(checkpoint)
            print("Generator downloaded")
            generator_loaded = True

    discriminator_loaded = False
    for key in discriminator_keys:
        if key in checkpoint:
            try:
                discriminator.load_state_dict(checkpoint[key])
                discriminator_loaded = True
                break
            except:
                continue
    
    if not discriminator_loaded:
        print("couldn't load discriminator")
    
    if os.path.exists(unet_model_path):
        unet_state_dict = torch.load(unet_model_path, map_location=device)

        if 'out.weight' in unet_state_dict:
            if unet_state_dict['out.weight'].shape[0] == 3:
                unet_state_dict['out.weight'] = unet_state_dict['out.weight'][:1]
                unet_state_dict['out.bias'] = unet_state_dict['out.bias'][:1]
        
        unet.load_state_dict(unet_state_dict, strict=False)
        print("UNet weights loaded (with adjustments)")
    else:
        print("no unet model")
    
    generator.eval()
    discriminator.eval()
    unet.eval()

    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    
    print(f"Image processing: {image_path}")
    image = Image.open(image_path).convert('RGB')
    original_size = image.size  # (width, height)
    
    image_tensor = transform(image).unsqueeze(0).to(device)

    anomaly_colors = {
        0: [0, 255, 0],    # Green 
        1: [255, 0, 0],    # Red
        2: [0, 0, 255],    # Blue  
        3: [255, 255, 0],  # Yellow
        4: [255, 0, 255],  # Purple
    }
    
    class_names = ['normal', 'abrasive', 'adhesive', 'cracks', 'delamination']

    with torch.no_grad():
        validity, class_probs = discriminator(image_tensor)
        predicted_class = torch.argmax(class_probs, dim=1).item()
        confidence = class_probs[0, predicted_class].item()
        
        z = torch.randn(1, 100).to(device)
        label_tensor = torch.tensor([predicted_class], device=device)
        reconstructed = generator(z, label_tensor)
        
        gan_difference = torch.abs(image_tensor - reconstructed)

        unet_mask = unet(image_tensor)
        
        print(f"Input shape: {image_tensor.shape}")
        print(f"UNet mask shape: {unet_mask.shape}")
        print(f"GAN difference shape: {gan_difference.shape}")
        
        if unet_mask.shape[1] != 1:
            unet_mask = unet_mask[:, :1]  # Take only first channel
        
        gan_diff_single_channel = gan_difference.mean(dim=1, keepdim=True)
        
        combined_anomaly = (unet_mask + gan_diff_single_channel) / 2
        
        anomaly_mask = (combined_anomaly > 0.5).float()
        
        anomaly_mask_np = anomaly_mask.squeeze().cpu().numpy()
        
        anomaly_mask_resized = cv2.resize(anomaly_mask_np, original_size)  # original_size is (width, height)
        
        gan_diff_visual = gan_difference.squeeze().mean(dim=0).cpu().numpy()
        unet_mask_visual = unet_mask.squeeze().cpu().numpy()
        combined_visual = combined_anomaly.squeeze().cpu().numpy()
    
    original_image = cv2.imread(image_path)
    original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    
    print(f"Original image shape: {original_image_rgb.shape}")
    print(f"Anomaly mask shape: {anomaly_mask_resized.shape}")
    
    color_mask = np.zeros_like(original_image_rgb)
    
    mask_bool = anomaly_mask_resized > 0
    
    for c in range(3):
        color_mask[:, :, c][mask_bool] = anomaly_colors[predicted_class][c]
    
    result_image = cv2.addWeighted(original_image_rgb, 0.7, color_mask, 0.3, 0)
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    axes[0].imshow(original_image_rgb)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    im2 = axes[1].imshow(gan_diff_visual, cmap='hot')
    axes[1].set_title('GAN Difference Map')
    plt.colorbar(im2, ax=axes[1])
    axes[1].axis('off')
    
    im3 = axes[2].imshow(unet_mask_visual, cmap='hot')
    axes[2].set_title('UNet Mask')
    plt.colorbar(im3, ax=axes[2])
    axes[2].axis('off')
    
    im4 = axes[3].imshow(combined_visual, cmap='hot')
    axes[3].set_title('Combined Anomaly Map')
    plt.colorbar(im4, ax=axes[3])
    axes[3].axis('off')
    
    axes[4].imshow(anomaly_mask_resized, cmap='gray')
    axes[4].set_title('Binary Anomaly Mask')
    axes[4].axis('off')
    
    axes[5].imshow(color_mask)
    axes[5].set_title('Color Mask')
    axes[5].axis('off')
    
    axes[6].imshow(result_image)
    axes[6].set_title(f'Result\nClass: {class_names[predicted_class]}\nConfidence: {confidence:.3f}')
    axes[6].axis('off')
    
    classes = list(range(len(class_names)))
    probs = class_probs[0].cpu().numpy()
    colors = ['green' if i == predicted_class else 'gray' for i in classes]
    axes[7].bar(classes, probs, color=colors)
    axes[7].set_xticks(classes)
    axes[7].set_xticklabels(class_names, rotation=45)
    axes[7].set_title('Class Probabilities')
    axes[7].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()
    
    print(f"\npredicted class: {class_names[predicted_class]}")
    print(f"confidence: {confidence:.3f}")
    print(f"size of the mask: {anomaly_mask_resized.shape}")
    
    print("\nprobability of classes:")
    for i, class_name in enumerate(class_names):
        prob = class_probs[0, i].item()
        print(f"  {class_name}: {prob:.4f}")

if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import os
import glob
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import random

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(SimpleUNet, self).__init__()
        
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)
        
        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        
        self.out = nn.Conv2d(16, 3, 1)
        
    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        
        b = self.bottleneck(p2)
        
        u2 = self.up2(b)
        if u2.shape != e2.shape:
            u2 = torch.nn.functional.interpolate(u2, size=e2.shape[2:], mode='bilinear')
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        if u1.shape != e1.shape:
            u1 = torch.nn.functional.interpolate(u1, size=e1.shape[2:], mode='bilinear')
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        return self.out(d1)




In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, num_classes=5, img_channels=3):
        super(Generator, self).__init__()
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        
        self.init_size = 16
        self.label_embedding = nn.Embedding(num_classes, latent_dim)
        
        self.l1 = nn.Sequential(nn.Linear(latent_dim * 2, 128 * self.init_size ** 2))
        
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.1),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64, 0.8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, img_channels, 3, stride=1, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        label_embed = self.label_embedding(labels)
        gen_input = torch.cat((label_embed, z), -1)
        
        out = self.l1(gen_input)
        out = out.view(out.shape[0], 128, self.init_size, self.init_size)
        img = self.conv_blocks(out)
        return img

class Discriminator(nn.Module):
    def __init__(self, num_classes=5, img_channels=3):
        super(Discriminator, self).__init__()
        
        def discriminator_block(in_filters, out_filters, bn=True):
            block = [nn.Conv2d(in_filters, out_filters, 3, 2, 1),
                    nn.LeakyReLU(0.2, inplace=True),
                    nn.Dropout2d(0.3)]
            if bn:
                block.append(nn.BatchNorm2d(out_filters, 0.8))
            return block
        
        self.model = nn.Sequential(
            *discriminator_block(img_channels, 16, bn=False),
            *discriminator_block(16, 32),
            *discriminator_block(32, 64),
            *discriminator_block(64, 128),
        )
        
        ds_size = 64 // 2 ** 4
        self.adv_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, 1), nn.Sigmoid())
        self.aux_layer = nn.Sequential(nn.Linear(128 * ds_size ** 2, num_classes), nn.Softmax(dim=1))
    
    def forward(self, img):
        out = self.model(img)
        out = out.view(out.shape[0], -1)
        validity = self.adv_layer(out)
        label = self.aux_layer(out)
        return validity, label




In [ ]:
class DefectDetector:
    def __init__(self, model_path, unet_path=None, img_size=64):
        self.img_size = img_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.generator = Generator(100, 5, 3).to(self.device)
        self.discriminator = Discriminator(5, 3).to(self.device)
        
        self.unet = SimpleUNet(in_channels=3, out_channels=1).to(self.device)
        
        checkpoint = torch.load(model_path, map_location=self.device)
        
        generator_loaded = False
        discriminator_loaded = False
        
        generator_keys = ['generator', 'G', 'generator_state_dict']
        discriminator_keys = ['discriminator', 'D', 'discriminator_state_dict']
        
        for key in generator_keys:
            if key in checkpoint:
                try:
                    self.generator.load_state_dict(checkpoint[key])
                    generator_loaded = True
                    break
                except:
                    continue
   
        for key in discriminator_keys:
            if key in checkpoint:
                try:
                    self.discriminator.load_state_dict(checkpoint[key])
                    print(f"Discriminator загружен из ключа: {key}")
                    discriminator_loaded = True
                    break
                except:
                    continue
        
        if unet_path:
            try:
                unet_checkpoint = torch.load(unet_path, map_location=self.device)
                
                if 'out.weight' in unet_checkpoint:
                    if unet_checkpoint['out.weight'].shape[0] == 3:
                        unet_checkpoint['out.weight'] = unet_checkpoint['out.weight'][:1]
                        if 'out.bias' in unet_checkpoint:
                            unet_checkpoint['out.bias'] = unet_checkpoint['out.bias'][:1]
                
                self.unet.load_state_dict(unet_checkpoint, strict=False)
                print(f"UNet загружен из: {unet_path}")
                self.unet_loaded = True
            except Exception as e:
                print(f"Ошибка загрузки UNet из {unet_path}: {e}")
                self.unet_loaded = False
        else:
            print("no unet, using only gan")
            self.unet_loaded = False
        
        self.generator.eval()
        self.discriminator.eval()
        self.unet.eval()

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)
        ])
        
        self.defect_names = {
            0: "normal(green)",
            1: "abrasive(red)",
            2: "adhesive(blue)",
            3: "crack(yellow)",
            4: "delamination(pink)"
        }
        
        # Цвета
        self.defect_colors = {
            0: [0, 255, 0],    
            1: [255, 0, 0],    
            2: [0, 0, 255],   
            3: [255, 255, 0],  
            4: [255, 0, 255]   
        }

    def preprocess_image(self, image_path):
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"couldn't load image: {image_path}")
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    def process_tiles(self, image, tile_size=64, stride=32):
        h, w = image.shape[:2]
        tiles = []
        positions = []
        
        for y in range(0, h, stride):
            for x in range(0, w, stride):
                y_end = min(y + tile_size, h)
                x_end = min(x + tile_size, w)
                
                if (y_end - y) < tile_size // 2 or (x_end - x) < tile_size // 2:
                    continue
                
                tile = image[y:y_end, x:x_end]
                
                if tile.shape[0] != tile_size or tile.shape[1] != tile_size:
                    tile = cv2.resize(tile, (tile_size, tile_size))
                
                tiles.append(tile)
                positions.append((x, y, x_end, y_end))
        
        return tiles, positions

    def analyze_tile(self, tile_rgb):
        pil_image = Image.fromarray(tile_rgb)
        image_tensor = self.transform(pil_image).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            validity, class_probs = self.discriminator(image_tensor)
            predicted_class = torch.argmax(class_probs, dim=1).item()
            confidence = class_probs[0, predicted_class].item()
            
            if predicted_class != 0 and confidence > 0.7:  
                z = torch.randn(1, 100).to(self.device)
                label_tensor = torch.tensor([predicted_class], device=self.device)
                reconstructed = self.generator(z, label_tensor)
                
                gan_diff = torch.abs(image_tensor - reconstructed).mean(dim=1, keepdim=True)
                
                if self.unet_loaded:
                    try:
                        unet_mask = self.unet(image_tensor)
                        if unet_mask.shape[1] != 1:
                            unet_mask = unet_mask[:, :1]
                        anomaly_map = (gan_diff + unet_mask) / 2
                    except:
                        anomaly_map = gan_diff
                else:
                    anomaly_map = gan_diff
                
                if anomaly_map.max() > 0:
                    anomaly_map = anomaly_map / anomaly_map.max()
                
                anomaly_np = anomaly_map.squeeze().cpu().numpy()
                
                return {
                    'class': predicted_class,
                    'confidence': confidence,
                    'anomaly_map': anomaly_np,
                    'is_defect': True
                }
            else:
                return {
                    'class': 0, 
                    'confidence': confidence,
                    'anomaly_map': None,
                    'is_defect': False
                }

    def detect_defects(self, image_path):
        original_rgb = self.preprocess_image(image_path)
        h, w = original_rgb.shape[:2]
        
        defect_mask_float = np.zeros((h, w, 3), dtype=np.float32)
        defect_counts = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
        defect_confidence_sum = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
        
        total_tiles = 0
        defect_tiles = 0
        
        tiles, positions = self.process_tiles(original_rgb, tile_size=self.img_size, stride=self.img_size//2)
        total_tiles = len(tiles)
        
        
        for tile, (x1, y1, x2, y2) in zip(tiles, positions):
            result = self.analyze_tile(tile)
            
            if result is not None:
                class_id = result['class']
                defect_counts[class_id] += 1
                defect_confidence_sum[class_id] += result['confidence']
                
                if result['is_defect']:
                    defect_tiles += 1
                    
                    if result['anomaly_map'] is not None:
                        anomaly_map = result['anomaly_map']
                        tile_h, tile_w = anomaly_map.shape
                        
                        if anomaly_map.max() > 0:
                            threshold = np.percentile(anomaly_map, 85)  
                            defect_pixels = anomaly_map > threshold
                            
                            if np.any(defect_pixels):
                                color = self.defect_colors[class_id]
                                
                                actual_h = y2 - y1
                                actual_w = x2 - x1
                                
                                if actual_h != tile_h or actual_w != tile_w:
                                    defect_pixels_resized = cv2.resize(
                                        defect_pixels.astype(np.float32), 
                                        (actual_w, actual_h)
                                    ) > 0.5
                                else:
                                    defect_pixels_resized = defect_pixels
                                
                                for c in range(3):
                                    defect_mask_float[y1:y2, x1:x2, c][defect_pixels_resized] += color[c]
        
        defect_mask = np.clip(defect_mask_float, 0, 255).astype(np.uint8)
        
        total_pixels = h * w
        defect_percentages = {}
        
        for class_id in range(1, 5):
            if class_id in self.defect_colors:
                color = np.array(self.defect_colors[class_id])
                if np.any(defect_mask > 0):
                    color_mask = np.all(np.abs(defect_mask - color) < 50, axis=2)
                    defect_counts[class_id] = np.sum(color_mask)
                    defect_percentages[class_id] = (defect_counts[class_id] / total_pixels) * 100
        
        if total_tiles > 0:
            defect_tile_percentage = (defect_tiles / total_tiles) * 100
            normal_tile_percentage = 100 - defect_tile_percentage
        else:
            defect_tile_percentage = 0
            normal_tile_percentage = 100
        
        total_defect_pixels = sum(defect_counts.get(i, 0) for i in range(1, 5))
        if total_pixels > 0:
            defect_pixel_percentage = (total_defect_pixels / total_pixels) * 100
            normal_pixel_percentage = 100 - defect_pixel_percentage
        else:
            defect_pixel_percentage = 0
            normal_pixel_percentage = 100
        
        return (original_rgb, defect_mask, defect_counts, defect_percentages, 
                normal_tile_percentage, defect_tile_percentage,
                normal_pixel_percentage, defect_pixel_percentage)

    def show_results(self, image_path):
        (original_rgb, defect_mask, defect_counts, defect_percentages, 
         normal_tile_percent, defect_tile_percent,
         normal_pixel_percent, defect_pixel_percent) = self.detect_defects(image_path)
        
        result_image = cv2.addWeighted(original_rgb, 0.7, defect_mask, 0.3, 0)
        
        print(f"\nshape: {self.img_size}x{self.img_size}):")
        print(f"  Normal area: {normal_tile_percent:.1f}%")
        print(f"  Defects area: {defect_tile_percent:.1f}%")
    
        
        print("\nDefects detalization:")
        found_defects = False
        for class_id in [2, 1, 3, 4]:  
            if class_id in defect_percentages and defect_percentages[class_id] > 0.01:
                name = self.defect_names[class_id]
                percentage = defect_percentages[class_id]
                count = defect_counts.get(class_id, 0)
                print(f"  {name}: {percentage:.2f}% ({count} pixels)")
                found_defects = True

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        
        axes[0].imshow(original_rgb)
        axes[0].set_title('Original image')
        axes[0].axis('off')
        
        axes[1].imshow(defect_mask)
        axes[1].set_title('Defects map')
        axes[1].axis('off')
        
        axes[2].imshow(result_image)
        axes[2].set_title('Result')
        axes[2].axis('off')
        
        if found_defects:
            defect_classes = []
            defect_values = []
            colors = []
            
            for class_id in [1, 2, 3, 4]:
                if class_id in defect_percentages and defect_percentages[class_id] > 0:
                    defect_classes.append(self.defect_names[class_id].split('(')[0])
                    defect_values.append(defect_percentages[class_id])
                    colors.append([c/255 for c in self.defect_colors[class_id]])
            
            if defect_values:
                axes[3].bar(defect_classes, defect_values, color=colors)
                axes[3].set_title('Defects(%)')
                axes[3].set_ylabel('Percentage of area')
                axes[3].tick_params(axis='x', rotation=45)
                axes[3].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return result_image, defect_mask, normal_pixel_percent, defect_pixel_percent

In [ ]:
detector = DefectDetector(
    model_path= r"C:\Users\Administrator\OneDrive\Desktop\Gan+Unet project\hybrid codes\final_hybrid_model.pth",
    unet_path= r"C:\Users\Administrator\OneDrive\Desktop\Gan+Unet project\hybrid codes\2_unet_trained.pth"
)
result, mask = detector.show_results(r"C:\Users\Administrator\OneDrive\Desktop\diana\Gan Anomaly\article_jpg2.jpg")